In [ ]:
from dijido import dawlla
import dijido

In [ ]:
dawlla.login()

In [ ]:
import dijido
from datetime import timedelta
from datetime import datetime
import pytz
import pandas
import numpy

from plotly import graph_objects
from plotly import offline as plotly

import plotly.io as pio
pio.renderers.default = "notebook"

import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

import scipy
from scipy import signal

In [ ]:
transactions = dawlla.get_transactions()
accounts = dawlla.get_accounts()
categories = dawlla.get_categories()

In [ ]:
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"
LOCAL_TIMEZONE = datetime.now().astimezone().tzinfo

In [ ]:
def plot_expenditure_by_interval(categories_of_interest, start_date, end_date, interval="month"):

    if interval == "month":
        pass
    else:
        raise NotImplementedError("One day")
    
    traces = []

    category_name_map = {
        x["name"]: x
        for x in dawlla.get_categories()
    }

    transactions_by_date = {}

    for transaction in transactions:
        transaction_date = dijido.convert_to_datetime(transaction["date"])
        transaction_date_string = dijido.local_date_string(transaction_date)[0:10]

        if transaction_date_string not in transactions_by_date:
            transactions_by_date[transaction_date_string] = [transaction]
        else:
            transactions_by_date[transaction_date_string].append(transaction)
        
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    data = []
    
    for category_name in categories_of_interest:

        category = category_name_map[category_name]
        category_id = category["_id"]
        previous_date = start_date
        date = start_date

        current_month = date.month
        current_month_date = date
    
        xs = []
        ys = []
        tickdates = []
        cumulative_expenditure = 0
    
        while date <= end_date:
    
            date_month = date.month
            date_string = start_date.strftime("%Y-%m-%d")

            # If we've ticked over a month, time to add data
            if date_month > current_month:
                
                # Calculate the numbers for this month
                num_days = (previous_date - current_month_date).days + 1
                days = (current_month_date - start_date).days
                xs.append(days)
                tickdates.append(current_month_date)
                ys.append(cumulative_expenditure)
                
                data.append({
                    "month": current_month_date,
                    "expenditure": ys[-1],
                    "category_name": category_name
                })
                
                # Increment the month
                cumulative_expenditure = 0
                current_month = date.month
                current_month_date = date
    
            date_string = date.strftime("%Y-%m-%d")
    
            if date_string in transactions_by_date:
                for transaction in transactions_by_date[date_string]:
                    if transaction["category_id"] == category_name_map[category_name]["_id"]:
                        cumulative_expenditure += transaction["amount"] * -1

            previous_date = date
            date = date + timedelta(days=1)

        # Calculate the numbers for this month
        num_days = (previous_date - current_month_date).days + 1
        days = (current_month_date - start_date).days
        xs.append(days)
        tickdates.append(current_month_date)
        ys.append(cumulative_expenditure)
                
        data.append({
            "month": current_month_date,
            "expenditure": ys[-1],
            "category_name": category_name
        })
        
        trace = graph_objects.Scatter(
            x=xs,
            y=ys,
            name=category_name,
            line={
                "width": 3
            }
        )
    
        traces.append(trace)
    
    layout = {
        "xaxis": {
            "tickmode": "array",
            "tickvals": [(date - start_date).days for date in tickdates],
            "ticktext": [x.strftime("%b %Y") for x in tickdates]
        },
        "yaxis": {
            "title": "$",
            "rangemode": "tozero"
        },
        "plot_bgcolor": "rgba(255, 255, 255, 0)",
        "paper_bgcolor": "rgba(255, 255, 255, 0)",
        "title": "Expenditure analysis",
        "width": 1000
    }
    
    
    figure = graph_objects.Figure(data=traces, layout=layout)
    
    plotly.iplot(figure)

    return data


In [ ]:
data = plot_expenditure_by_interval(["Snacks", "Drinks", "Eating Out"], START_DATE, END_DATE, interval="month")

In [ ]:
pandas.DataFrame.from_records(data).to_csv("personal_food_expenditures.csv")